# LangChain 심화 : LCEL과 Runnable

이번 실습에서는 기초 파일에서 배운 Model, Prompt, Output Parser를 연결해 재사용 가능한 실행 흐름을 만듭니다.

**학습 목표**

> 1. 이번 실습에서는 LCEL(LangChain Expression Language)을 사용해 Prompt, Model, Output Parser를 하나의 실행 체인으로 연결하는 법을 익힌다.
> 2. `prompt | model | parser` 구조를 통해 데이터가 왼쪽에서 오른쪽으로 흐르는 LangChain 체인 구성 방식을 이해한다.
> 3. Runnable의 공통 실행 방식인 `invoke`, `batch`, `stream`을 사용해 단일 실행, 대량 실행, 스트리밍 실행을 연습한다.
> 4. **LCEL과 Runnable을 조립해 실무형 LLM 워크플로우를 만드는 법**을 익힌다.

# 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.

In [28]:
# 필요한 라이브러리 설치
%pip install -U langchain langchain-core langchain-openai python-dotenv pydantic pandas

Note: you may need to restart the kernel to use updated packages.


## (2) 라이브러리 Import

이번 실습에서 사용하는 핵심 객체는 다음과 같습니다.

| 객체 | 역할 |
|---|---|
| `ChatOpenAI` | OpenAI 채팅 모델을 LangChain 방식으로 호출 |
| `PromptTemplate` | 문자열 기반 프롬프트 템플릿 |
| `ChatPromptTemplate` | system/human 메시지를 분리하는 채팅 프롬프트 |
| `StrOutputParser` | 모델 응답에서 문자열만 추출 |
| `RunnableLambda` | 일반 파이썬 함수를 체인에 연결 |
| `RunnableParallel` | 여러 체인을 동시에 실행 |


In [29]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import(RunnableLambda,
                                     RunnableSequence,
                                     RunnableParallel)

## (3) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [30]:
load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## (4) 모델 준비

심화 실습은 체인을 구성하는 데 집중하므로 모델은 바로 준비합니다.

In [31]:
model = ChatOpenAI(
    model='gpt-4.1-mini',   # 사용할 OpenAI 모델 이름
    timeout=30,             # 응답 대기 시간 제한 : 30초
    max_retries=3,          # 요청 실패 시 최대 3번까지 재시도
)

parser = StrOutputParser()

model

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001CA2A832780>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001CA2A832EA0>, root_client=<openai.OpenAI object at 0x000001CA2A831E00>, root_async_client=<openai.AsyncOpenAI object at 0x000001CA2A832C40>, model_name='gpt-4.1-mini', model_kwargs={

# 2. Chain과 LCEL

- LCEL(LangChain Expression Language) : Runnable 객체를 `|` 파이프 연산자를 사용해 컴포넌트를 연결하고, 체인을 **선언적으로 정의**하는 표현 방식


이 구조에서 데이터는 왼쪽에서 오른쪽으로 흐릅니다.

| 단계 | 입력 | 출력 |
|---|---|---|
| Prompt | dict | PromptValue 또는 메시지 |
| Model | PromptValue 또는 메시지 | AIMessage |
| Parser | AIMessage | 문자열 또는 구조화 데이터 |

LCEL을 쓰면 프롬프트, 모델, 파서를 따로 실행하지 않고 하나의 실행 단위처럼 다룰 수 있습니다.

### 다이어그램

```mermaid
flowchart LR
    IN(["📥 dict input<br/>{role, question}"]):::input
    PT["📝 Prompt<br/>ChatPromptTemplate<br/>{role} · {question}"]:::node
    LM["🤖 LLM<br/>init_chat_model<br/>→ AIMessage"]:::node
    PA["🔍 Parser<br/>StrOutputParser<br/>→ str"]:::node
    OUT(["📤 최종 문자열"]):::output

    IN --> PT --> LM --> PA --> OUT

    classDef input  fill:#4f46e5,stroke:#3730a3,color:#fff
    classDef node   fill:#1e293b,stroke:#475569,color:#e2e8f0
    classDef output fill:#059669,stroke:#047857,color:#fff
```

코드 표현은 다음과 같이 간결합니다.

```
prompt | llm | parser
```

### 파이프 연산자의 이점

1. **가독성**: "프롬프트 → 모델 → 파서"의 처리 흐름이 코드에 그대로 드러납니다.
2. **재사용성**: `prompt | llm` 부분을 변수로 분리하면 서로 다른 파서와 조합할 수 있습니다.
3. **공통 인터페이스**: 각 Runnable은 `.invoke()`, `.stream()`, `.batch()`를 자동 지원합니다.

In [32]:
concept_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 수학을 쉽게 설명해주는 수학 강사이다.'),
    ('human', '개념: {concept}, 난이도: {level}, 설명, 예시, 주의점을 작성해줘.')
])

concept_chain = concept_prompt | model | parser

result = concept_chain.invoke({
    'concept': '미분',
    'level': '입문'
})

print(result)

개념: 미분

미분은 함수의 변화율을 구하는 과정입니다. 쉽게 말해, 어떤 값이 얼마나 빠르게 변하는지를 알아보는 방법이에요. 예를 들어, 자동차가 달릴 때 속도가 얼마나 빠르게 변하는지를 미분을 통해 알 수 있습니다.

설명:

함수 y = f(x)가 있을 때, x가 아주 조금 변할 때 y가 어떻게 변하는지를 보는 것이 미분입니다. 더 정확하게는 x의 변화량(Δx)이 0에 가까워질 때 y의 변화량(Δy)과 x의 변화량의 비율 Δy/Δx가 한 점에서 가지는 극한값을 구합니다. 이 극한값을 ‘미분계수’라고 하고, 함수 y의 미분을 f '(x) 또는 dy/dx로 표기합니다.

예시:

함수 y = x²를 미분해보겠습니다.

1. Δy = (x + Δx)² - x² = x² + 2xΔx + (Δx)² - x² = 2xΔx + (Δx)²  
2. Δy/Δx = 2x + Δx  
3. Δx → 0일 때, Δy/Δx → 2x

그래서 y = x²의 미분은 f '(x) = 2x 입니다.

주의점:

- 미분은 함수가 연속이고 충분히 부드러울 때만 가능합니다. 즉, 함수 그래프가 끊어지거나 뾰족한 점에서는 미분이 불가능할 수 있습니다.
- 미분 결과는 함수가 증가하는 속도(기울기)를 나타내지만, 이것이 항상 함수값의 크기와 동일하지는 않습니다. 예를 들어 y = x²에서 x=2일 때 함수값은 4지만, 미분값은 4입니다(기울기 4), 이 둘을 구분하세요.
- Δx를 0으로 ‘바로 대입하는 것’이 아니라, 극한을 이용한 정의임을 기억하세요. 

이해하기 쉽도록 간단한 함수부터 차근차근 미분 연습을 해보면 미분 개념을 익히는 데 도움이 됩니다!


## LCEL을 쓰는 이유

LCEL의 장점은 단순히 코드가 짧아지는 것이 아닙니다. 모든 체인이 공통 실행 인터페이스를 가지게 됩니다.

| 실행 방식 | 설명 |
|---|---|
| `invoke()` | 하나의 입력 실행 |
| `batch()` | 여러 입력을 한 번에 실행 |
| `stream()` | 결과를 생성되는 대로 출력 |

즉, 작은 예제로 만든 체인을 대량 처리나 스트리밍으로 쉽게 확장할 수 있습니다.

# 3. Runnable 실행 방식

- Runnable은 LangChain에서 "실행 가능한 객체"
- Prompt, Model, Parser, 직접 만든 함수까지 모두 Runnable처럼 연결할 수 있음

## (1) `invoke`

가장 기본적인 단일 실행입니다.

In [33]:
single_result = concept_chain.invoke({
    'concept':'벡터',
    'level':'고급'
})

print(single_result)

알겠습니다! 이번에는 고급 난이도의 벡터 개념에 대해 쉽게 설명해드리겠습니다.

---

### 벡터란?

벡터(Vector)는 크기와 방향을 모두 가진 수학적 객체입니다. 단순히 크기만 있는 스칼라와 달리, 벡터는 공간에서의 위치나 힘, 속도 등 방향이 중요한 상황을 표현할 때 사용합니다.

---

### 고급 개념 설명

1. **벡터 공간(Vector Space)**  
   벡터는 단순히 2차원이나 3차원 좌표평면상의 점을 나타내는 것뿐 아니라, 추상적인 벡터 공간에서 다양한 차원과 형태로 확장됩니다. 벡터 공간은 다음 조건을 만족하는 집합과 연산으로 구성됩니다.  
   - 벡터 덧셈이 닫혀 있음 (두 벡터의 합도 벡터)  
   - 스칼라 곱셈이 닫혀 있음  
   - 덧셈의 교환법칙, 결합법칙 등 기본 법칙을 만족

   예를 들어 함수들, 다차원 배열(텐서), 다항식 집합 등도 벡터 공간이 될 수 있습니다.

2. **내적과 외적**  
   - **내적(Dot product)**: 두 벡터의 관계를 수치로 나타내며, 두 벡터가 이루는 각도를 구하거나 벡터의 길이를 계산, 직교성 판단 등에 사용됩니다.  
   - **외적(Cross product)**: 3차원 공간에서 두 벡터에 수직인 벡터를 구할 때 쓰이며, 물리학에서 토크, 자기장 계산 등에 활용됩니다.

3. **선형 독립성과 기저(Basis)**  
   여러 벡터들의 집합이 각각 서로 선형 결합으로 표현되지 않는다면, 그 집합을 선형 독립이라고 합니다.  
   선형 독립인 벡터 집합을 기저라고 하며, 기저를 통해 벡터 공간 모든 벡터를 유일하게 표현할 수 있습니다.

4. **행렬과 벡터 변환**  
   벡터는 행렬과 곱하여 선형 변환을 수행할 수 있습니다. 이는 벡터 공간을 다른 벡터 공간으로 변환하는 방법을 제공하며, 회전, 이동, 확대/축소 등의 연산을 표현합니다.

---

### 예시

- 3차원 벡터 \(\mathbf{a} = (1, 2, 3)\)와 \(\mathbf{

## (2) `batch`

여러 입력을 같은 체인으로 한 번에 처리합니다. 고객 후기 분석, 문서 요약, 강의 주제별 설명 생성처럼 반복 작업에 유용합니다.

In [34]:
topics = [
    {'concept':'집합', 'level':'입문'},
    {'concept':'확률', 'level':'중급'},
    {'concept':'적분', 'level':'고급'}
]

batch_results = concept_chain.batch(topics)

for topic, output in zip(topics, batch_results):
    print('='*110)
    print(topic['concept'])
    print(output[:500])

집합
개념: 집합

설명:  
집합이란 서로 구별되는 원소들의 모임을 말합니다. 쉽게 말해, 어떤 대상들을 한데 모아 놓은 '묶음'이라고 생각하면 됩니다. 집합 안에 들어가는 각각의 대상은 '원소'라고 부릅니다. 예를 들어, '사과, 바나나, 오렌지'라는 과일들의 모임도 하나의 집합이 됩니다.

예시:  
- A = {1, 2, 3}  
- B = {사과, 바나나, 오렌지}  
- C = {x | x는 1부터 5까지의 자연수}  
위 예시에서 {} 안에 있는 것들이 집합의 원소입니다.

주의점:  
1. 집합에서 같은 원소가 여러 번 있어도 한 번만 센다. 예를 들어, {1, 1, 2}는 {1, 2}와 같습니다.  
2. 원소의 순서는 중요하지 않다. {1, 2, 3}과 {3, 2, 1}은 같은 집합이다.  
3. 집합을 표현할 때, 중복 없이 원소를 나열하고, 반드시 중괄호{}를 사용해야 한다.
확률
좋습니다! 확률에 대해 중급 수준으로 쉽게 설명해드릴게요.

---

### 개념: 확률 (Probability)

확률은 어떤 사건이 일어날 가능성을 수치로 나타낸 것입니다. 0부터 1까지의 값을 가지며, 0은 절대 일어나지 않는 사건, 1은 확실히 일어나는 사건을 의미해요. 예를 들어, 동전을 던졌을 때 앞면이 나올 확률은 0.5입니다.

---

### 설명

확률을 구할 때는 보통 다음과 같은 식을 사용해요:

\[
\text{확률} = \frac{\text{원하는 경우의 수}}{\text{전체 경우의 수}}
\]

- **전체 경우의 수**: 어떤 실험에서 일어날 수 있는 가능한 모든 결과의 수입니다.
- **원하는 경우의 수**: 관심 있는 특정 사건이 발생하는 경우의 수입니다.

확률은 항상 0 이상 1 이하이고, 모든 가능한 사건의 확률을 다 더하면 1이 됩니다.

---

### 예시

**예시 1.**  
주사위를 던졌을 때, 4가 나올 확률은?

- 전체 경우의 
적분
물론입니다! 고급 수준의 적분 개념에 대해 쉽게 설명해드릴게요.

---


## (3) `stream`

`stream()`은 결과가 생성되는 동안 조각을 순서대로 받을 수 있습니다. 챗봇의 타이핑 효과처럼 사용자 경험을 좋게 만들 때 사용합니다.

In [35]:
for chunk in concept_chain.stream({
    "concept":"행렬",
    "level":"입문"
}):
    print(chunk, end="", flush=True)    # flush=True 출력 버퍼를 바로 비워서, 답변이 실시간으로 타이핑 되도록

물론입니다! 행렬에 대해 쉽고 친절하게 설명해드릴게요.

---

### 개념: 행렬 (Matrix)

행렬은 숫자들을 가로와 세로로 직사각형 모양으로 배열한 표라고 생각하면 됩니다. 숫자들이 일정한 규칙에 따라 행(row)과 열(column)로 나누어져 있어서, 여러 데이터를 정리하거나 계산할 때 아주 유용합니다.

예를 들어, 2행 3열 행렬은 이렇게 생겼어요:

\[
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{bmatrix}
\]

위 행렬에서 1과 2, 3이 첫 번째 행, 4, 5, 6이 두 번째 행이며, 1과 4는 첫 번째 열, 2와 5는 두 번째 열, 3과 6은 세 번째 열에 위치해 있습니다.

---

### 예시

1. 학생들의 시험 점수를 행렬로 표현하기

| 학생 | 수학 | 영어 | 과학 |
|-------|------|------|------|
| 철수  | 85   | 90   | 78   |
| 영희  | 92   | 88   | 95   |

이 표를 숫자만 모아서 행렬로 바꾸면

\[
\begin{bmatrix}
85 & 90 & 78 \\
92 & 88 & 95
\end{bmatrix}
\]

가 됩니다. 이렇게 하면 점수 데이터를 한눈에 볼 수 있죠.

2. 행렬끼리의 덧셈

\[
A = \begin{bmatrix}1 & 2 \\ 3 & 4\end{bmatrix}, \quad B = \begin{bmatrix}5 & 6 \\ 7 & 8\end{bmatrix}
\]

두 행렬 \(A\)와 \(B\)를 더하면 각 자리 숫자끼리 더해서

\[
A + B = \begin{bmatrix}1+5 & 2+6 \\ 3+7 & 4+8\end{bmatrix} = \begin{bmatrix}6 & 8 \\ 10 & 12\end{bmatrix}
\]

---

### 주의점

1. **크기가 같아야 한다**  
   행렬끼리 더하거나 뺄 때는 꼭 행과 열의 개수가 똑같아야 합니다. 크기가 다르면 계산할 수 없어요.

# 4. Runnable 컴포넌트

LCEL 체인을 구성하는 모든 부품은 공통적으로 `Runnable` 인터페이스를 따릅니다. 보다 복잡한 처리 흐름을 구성할 때 다음 유틸리티 컴포넌트를 활용합니다.

| 컴포넌트 | 역할 |
|---|---|
| `RunnableSequence` | 파이프라인 (`\|` 연산자로 자동 생성) |
| `RunnableLambda` | 일반 파이썬 함수를 Runnable로 래핑 |
| `RunnableParallel` | 여러 체인을 병렬 실행 |

## 4.1. RunnableSequence
- LCEL의 가장 기본적인 구성

In [36]:
prompt = ChatPromptTemplate.from_messages([
    ("system","사용자가 입력한 요리의 레시피를 생각해 주세요."),
    ("human", "{dish}")
    ])

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

output_parser = StrOutputParser()

In [37]:
# promt->model->output_parser 순으로 invoke().
prompt_value = prompt.invoke({"dish":"카레"})
ai_message = model.invoke(prompt_value)
output = output_parser.invoke(ai_message)

print(output)

카레 레시피를 알려드릴게요!

[재료]
- 닭고기 또는 돼지고기 300g
- 감자 2개
- 당근 1개
- 양파 1개
- 카레 가루 3~4 큰술 (또는 시판 카레 블록 1팩)
- 식용유 2 큰술
- 물 600ml
- 소금, 후추 약간

[만드는 법]
1. 재료 손질: 감자, 당근, 양파는 껍질을 벗기고 한 입 크기로 썰어주세요. 고기도 먹기 좋은 크기로 썰어주세요.
2. 팬에 식용유를 두르고 중불에서 양파를 투명해질 때까지 볶아주세요.
3. 고기를 넣고 겉면이 익을 때까지 볶아주세요.
4. 감자와 당근을 넣고 함께 볶아주세요.
5. 물을 붓고 끓기 시작하면 중약불로 줄여 재료가 부드러워질 때까지 약 15~20분간 끓여주세요.
6. 카레 가루나 카레 블록을 넣고 잘 저어가며 녹여주세요.
7. 소금과 후추로 간을 맞추고 5분 정도 더 끓여 완성합니다.

밥과 함께 맛있게 드세요! 필요하시면 채소나 고기 종류를 바꾸어도 좋아요.


In [38]:
# LCEL 사용. | 연산자 사용해서 체인으로 연결.
chain = prompt | model | output_parser

output = chain.invoke({"dish":"카레"})
print(output)

카레 레시피를 알려드릴게요!

[재료]
- 닭고기 또는 돼지고기 300g
- 감자 2개
- 당근 1개
- 양파 1개
- 카레 가루 3~4 큰술 (또는 시판 카레 블록 1팩)
- 식용유 2 큰술
- 물 600ml
- 소금, 후추 약간

[만드는 법]
1. 재료 손질: 감자, 당근, 양파는 껍질을 벗기고 한 입 크기로 썰어주세요. 고기도 먹기 좋은 크기로 썰어주세요.
2. 팬에 식용유를 두르고 중불에서 양파를 투명해질 때까지 볶아주세요.
3. 고기를 넣고 겉면이 익을 때까지 볶아주세요.
4. 감자와 당근을 넣고 함께 볶아주세요.
5. 물을 붓고 끓기 시작하면 중약불로 줄여 재료가 부드러워질 때까지 약 15~20분간 끓여주세요.
6. 카레 가루 또는 카레 블록을 넣고 잘 저어가며 녹여주세요.
7. 약 5분간 더 끓여 농도를 맞추고, 소금과 후추로 간을 맞추면 완성입니다.

밥과 함께 맛있게 드세요!


## 4.2. RunnableLambda

`RunnableLambda`는 일반 파이썬 함수를 LangChain 체인 안에 넣을 때 사용합니다.

예를 들어 모델 답변 뒤에 자동 안내 문구를 붙이거나, 출력 문자열을 후처리하거나, 입력값을 정리하는 함수를 체인에 연결할 수 있습니다.

In [39]:
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful assistant."),
    ("human", "{input}")
    ])

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

output_parser = StrOutputParser()

In [40]:
def upper(text:str) -> str:
    return text.upper()

chain = prompt | model | output_parser | RunnableLambda(upper)

ai_message = chain.invoke({'input': "Hello!!"})

print(ai_message)

HELLO! HOW CAN I ASSIST YOU TODAY?


### chain 데코레이터를 사용한 RunnableLambda 구현

In [41]:
from langchain_core.runnables import chain

# @데코레이터 : 함수를 감싸서 기능을 추가하거나 형태를 바꾸는 문법
# upper는 그냥 파이썬 함수. @chain 붙이면 랭체인이 이 함수를 runnable처럼 실행 가능한 체인 요소로 바꿔줌.
@chain
def upper(text:str) -> str:
    return text.upper()

chain = prompt | model | output_parser | upper

ai_message = chain.invoke({'input': "Hello!!"})

print(ai_message)

HELLO! HOW CAN I ASSIST YOU TODAY?


### RunnableLambda 자동 변환


In [42]:
# 위의 코드와 차이 : upper 함수를 랭체인 Runnable로 명시적으로 바꾸느냐, 자동 변환에 맡기느냐.
# 랭체인 LCEL에서는 일반 함수르 파이프라인에 연결하면 내부적으로 자동으로 RunnableLambda처럼 감싸줌
def upper(text:str) -> str:
    return text.upper()

chain = prompt | model | output_parser | upper

In [43]:
ai_message = chain.invoke({'input':'Hello!!'})

print(ai_message)

HELLO! HOW CAN I ASSIST YOU TODAY?


### Runnable의 입력 타입과 출력 타입에 주의


In [44]:
def upper(text:str) -> str:
    return text.upper()

chain = prompt | model | upper

In [67]:
ai_message = chain.invoke({'input': "Hello!!"})

print(ai_message)

AttributeError: 'AIMessage' object has no attribute 'upper'

In [68]:
"""
model의 출력은 AIMessage 객체인데,
output_parser가 체인 사이에 없어 문자열로 변환되지 않은 채
upper()에 전달되어 .upper() 호출 오류가 발생했습니다.
"""

'\nmodel의 출력은 AIMessage 객체인데,\noutput_parser가 체인 사이에 없어 문자열로 변환되지 않은 채\nupper()에 전달되어 .upper() 호출 오류가 발생했습니다.\n'

## [실습] 후처리 함수 만들기

모델 답변의 앞뒤 공백을 제거하고, 맨 앞에 `[강의 메모]`를 붙이는 함수를 작성해 봅니다.

In [47]:
def format_lecture_memo(text: str) -> str:
    cleaned_text = text.strip()
    return f'[강의 메모]\n{cleaned_text}'

memo_chain = concept_chain | RunnableLambda(format_lecture_memo)
print(memo_chain.invoke({"concept": "RunnableLambda", "level": "입문"}))

[강의 메모]
개념: RunnableLambda  
RunnableLambda는 자바(Java)에서 람다식(lambda expression)을 사용하여 Runnable 인터페이스를 구현하는 방법입니다. Runnable은 실행할 코드를 정의하는 함수형 인터페이스로, 반드시 실행할 코드를 담은 `run()` 메서드를 하나 가지고 있습니다. 람다식을 사용하면 익명 클래스를 작성하는 번거로움 없이 간결하게 Runnable을 구현할 수 있습니다.

---

### 설명  
- **Runnable 인터페이스**는 인자가 없고 반환값도 없는 `void run()` 메서드를 갖고 있어, 스레드나 다른 작업으로 실행할 코드를 정의할 때 자주 쓰입니다.  
- 자바 8부터 지원된 **람다식**은 함수형 인터페이스를 간단하게 구현할 수 있는 문법입니다. Runnable도 함수형 인터페이스이므로 람다로 쉽게 표현할 수 있습니다.  
- 람다식을 쓰면 간결하고 가독성이 좋은 코드를 쓸 수 있어 스레드 작업이나 비동기 작업을 구현할 때 편리합니다.

---

### 예시

```java
public class RunnableLambdaExample {
    public static void main(String[] args) {
        // 람다식을 이용해 Runnable 구현
        Runnable runnableLambda = () -> {
            for (int i = 1; i <= 5; i++) {
                System.out.println("람다 스레드 실행 중: " + i);
                try {
                    Thread.sleep(500); // 0.5초 쉬기
                } catch (InterruptedException e) {
                    e.printStackTrace();
                }
            }
  

## 4.3. RunnableParallel

`RunnableParallel`은 같은 입력을 여러 체인에 동시에 전달하고, 결과를 딕셔너리로 모아줍니다.

In [48]:
model = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0
)

output_parser = StrOutputParser()

In [51]:
optimistic_prompt = ChatPromptTemplate.from_messages([
    ('system','당신은 낙관주의자입니다. 사용자의 입력에 대해 낙관적인 의견을 제공하세요.'),
    ('human', '{topic}')
])

optimistic_chain = optimistic_prompt | model | output_parser

pessimistic_prompt = ChatPromptTemplate.from_messages([
    ('system','당신은 비관주의자입니다. 사용자의 입력에 대해 비관적인 의견을 제공하세요.'),
    ('human', '{topic}')
])

pessimistic_chain = optimistic_prompt | model | output_parser

In [ ]:
import pprint

parallel_chain = RunnableParallel({
    'optimistic_opinion': optimistic_chain,
    'pessimistic_opinion': pessimistic_chain
})

output = parallel_chain.invoke({'topic':'취업할 수 있을까?'})
pprint.pprint(output)

{'optimistic_opinion': '물론이죠! 당신의 노력과 긍정적인 마음가짐이 있다면 분명 좋은 기회가 찾아올 거예요. 지금까지 '
                       '쌓아온 경험과 능력을 믿고 꾸준히 도전한다면 원하는 직장을 꼭 얻을 수 있을 거라고 확신합니다. '
                       '힘내세요!',
 'pessimistic_opinion': '물론이죠! 당신의 노력과 긍정적인 태도라면 분명 좋은 기회가 찾아올 거예요. 지금까지 쌓아온 '
                        '경험과 능력을 믿고 꾸준히 도전한다면 원하는 취업에 꼭 성공할 수 있을 거라고 확신합니다. '
                        '힘내세요!'}


### RunnableParallel의 출력을 Runnable의 입력으로 연결하기


In [56]:
synthesize_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 객관적인 AI입니다. 두 가지 의견을 종합하세요.'),
    ('human', '낙관적인 의견: {optimistic_opinion}, 비관적인 의견: {pessimistic_opinion}')
])

In [60]:
model = ChatOpenAI(
    model='gpt-5.5',
    timeout=30,
    max_retries=3
)

synthesize_chain = (
    RunnableParallel(
    {
    "optimistic_opinion" : optimistic_chain,
    "pessimistic_opinion" : pessimistic_chain
    }
    )
    | synthesize_prompt
    | model
    | output_parser
)

output = synthesize_chain.invoke({'topic': "대한민국은 월드컵 16강 올라갈 수 있을까?"})
pprint.pprint(output)

('두 의견을 종합하면, **대한민국의 월드컵 16강 진출 가능성은 충분히 기대해볼 만하지만, 조 편성·상대 전력·선수 컨디션 같은 변수에 '
 '따라 달라질 수 있다**고 정리할 수 있습니다.\n'
 '\n'
 '제시된 낙관적 의견과 비관적 의견의 내용이 동일하게 긍정적인 관점을 담고 있기 때문에, 실제로는 두 의견 모두 한국 축구의 성장 가능성과 '
 '경쟁력을 높게 평가하고 있습니다. 한국은 과거에도 월드컵 16강에 진출한 경험이 있고, 유럽 무대에서 활약하는 선수들과 조직력을 바탕으로 '
 '국제무대에서 경쟁력을 보여준 바 있습니다. 팬들의 응원 역시 선수들에게 긍정적인 동기부여가 될 수 있습니다.\n'
 '\n'
 '다만 객관적으로 보면 월드컵은 조 추첨 결과, 상대 팀의 수준, 부상 여부, 경기 당일의 전술과 집중력 등 많은 변수가 작용합니다. '
 '따라서 16강 진출을 낙관할 근거는 있지만, 확정적으로 보기보다는 **가능성 있는 목표**로 보는 것이 균형 잡힌 평가입니다.')


### RunnableParallel 자동 변환

In [65]:
synthesize_chain = (
    {
    'optimistic_opinion': optimistic_chain,
    'pessimistic_opinion': pessimistic_chain
    }
    | synthesize_prompt
    | model
    | output_parser
)

In [66]:
output = synthesize_chain.invoke({'topic': "대한민국은 월드컵 16강 올라갈 수 있을까?"})
pprint.pprint(output)

('두 의견을 종합하면, 제시된 “낙관적 의견”과 “비관적 의견” 모두 실제 내용은 대한민국의 월드컵 16강 진출 가능성을 긍정적으로 보고 '
 '있습니다. 따라서 두 의견 사이에 실질적인 대립은 크지 않고, 공통적으로 한국 축구의 성장과 선수층, 팀워크, 팬들의 응원을 강점으로 '
 '평가하고 있습니다.\n'
 '\n'
 '객관적으로 보면 대한민국의 16강 진출 가능성은 **충분히 기대해볼 만하지만, 확실하다고 단정하기는 어렵습니다.** 한국은 최근 꾸준히 '
 '국제무대 경험을 쌓았고, 해외 무대에서 활약하는 선수들과 젊은 자원들이 늘어나면서 경쟁력이 강화되었습니다. 조직력과 빠른 전환, 강한 '
 '정신력 역시 월드컵 같은 단기전에서 중요한 장점이 될 수 있습니다.\n'
 '\n'
 '다만 16강 진출 여부는 조 편성, 상대 팀 전력, 핵심 선수들의 부상 여부, 경기 당일의 컨디션과 전술 대응에 크게 좌우됩니다. 특히 '
 '월드컵에서는 한두 경기의 결과가 전체 흐름을 결정하기 때문에, 실력뿐 아니라 운과 변수도 무시할 수 없습니다.\n'
 '\n'
 '종합하면, 대한민국은 16강에 도전할 만한 전력을 갖추고 있으며 긍정적인 전망도 가능하지만, 이를 현실로 만들기 위해서는 안정적인 수비, '
 '결정력, 상대별 맞춤 전략이 뒷받침되어야 합니다. **낙관은 가능하되, 신중한 기대가 가장 균형 잡힌 평가**라고 볼 수 있습니다.')


## [실습] 주식 분석 리포트 생성기

이제 기본 파트의 Model, Prompt, Output Parser를 연결해 금융/주식 분석용 작은 프로젝트를 만듭니다.

이번 예시는 실시간 시세 조회나 투자 추천이 아니라, **제공된 예시 정보만 바탕으로 주식 분석 리포트 형식을 연습하는 실습**입니다.

## 목표

하나의 종목 또는 기업 정보를 입력하면 다음 내용을 구조화해서 생성합니다.

| 필드 | 설명 |
|---|---|
| `ticker` | 분석 대상 종목 또는 기업명 |
| `business_summary` | 기업과 업종에 대한 요약 |
| `investment_points` | 투자 관점에서 볼 만한 긍정 요인 |
| `risk_factors` | 확인해야 할 리스크 요인 |
| `key_indicators` | 분석할 때 참고할 핵심 지표 |
| `analyst_memo` | 투자 판단 전 확인해야 할 애널리스트 메모 |


In [73]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

model = ChatOpenAI(
    model="gpt-4.1-mini",
    timeout=120,
    max_retries=3,
)

output_parser = StrOutputParser()

ticker_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주식 분석 리포트를 작성해주는 30년차 주식 전문가이다. 사용자의 입력 종목명을 분석 대상 종목 또는 기업명으로 정리하세요."),
    ("human", "분석 대상: {ticker}")
])
ticker_chain = ticker_prompt | model | output_parser

business_summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 입력 종목에 대해 기업과 업종 요약을 작성하세요."),
    ("human", "분석 대상: {ticker}")
])
business_summary_chain = business_summary_prompt | model | output_parser

investment_points_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 입력 종목에 대해 투자 관점에서 볼 만한 긍정 요인을 작성하세요."),
    ("human", "분석 대상: {ticker}")
])
investment_points_chain = investment_points_prompt | model | output_parser

risk_factors_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 입력 종목에 대해 확인해야 할 리스크 요인을 작성하세요."),
    ("human", "분석 대상: {ticker}")
])
risk_factors_chain = risk_factors_prompt | model | output_parser

key_indicators_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 입력 종목을 분석할 때 참고할 핵심 지표를 작성하세요."),
    ("human", "분석 대상: {ticker}")
])
key_indicators_chain = key_indicators_prompt | model | output_parser

analyst_memo_prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 입력 종목에 대해 투자 판단 전 확인해야 할 애널리스트 메모를 작성하세요."),
    ("human", "분석 대상: {ticker}")
])
analyst_memo_chain = analyst_memo_prompt | model | output_parser

report_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주식 분석 리포트를 작성해주는 30년차 주식 전문가이다. 아래 분석 내용을 종합해 하나의 리포트로 작성하세요."),
    ("human", """
분석 대상:
{ticker}

기업/업종 요약:
{business_summary}

투자 포인트:
{investment_points}

리스크 요인:
{risk_factors}

핵심 지표:
{key_indicators}

애널리스트 메모:
{analyst_memo}

위 내용을 바탕으로 주식 분석 리포트를 작성해줘.
""")
])

synthe_chain = (
    RunnableParallel({
        "ticker": ticker_chain,
        "business_summary": business_summary_chain,
        "investment_points": investment_points_chain,
        "risk_factors": risk_factors_chain,
        "key_indicators": key_indicators_chain,
        "analyst_memo": analyst_memo_chain,
    })
    | report_prompt
    | model
    | output_parser
)

result = synthe_chain.invoke({
    "ticker": "한미반도체"
})

print(result)

[한미반도체 주식 분석 리포트]

1. 기업 개요
한미반도체는 한국을 기반으로 반도체 및 디스플레이 제조용 장비를 연구·개발·생산하는 전문 기업입니다. 웨이퍼 반송 장비, 플라즈마 장비, 노광장비 등 후공정 장비를 주력으로 하며, 첨단 공정 기술에 부합하는 다양한 제품을 제공하고 있습니다. 특히 글로벌 반도체 시장의 성장과 맞물려 기술 혁신과 해외시장 진출에 집중하여 경쟁력을 확대 중입니다.

2. 산업 및 시장 동향
반도체 장비 산업은 첨단 기술과 고도의 품질 요구가 필수적인 분야로, AI, 5G, 전기차 등 신산업 부문의 성장에 힘입어 반도체 수요가 꾸준히 증가하고 있습니다. 이에 따라 신규 투자 및 설비 확장이 이어져 장비 수요 역시 확대되는 긍정적 환경에 있습니다. 다만, 미·중 무역 분쟁, 글로벌 공급망 문제, 반도체 경기 사이클 변동성 등 불확실성 요인이 상존합니다.

3. 투자 포인트
- 글로벌 반도체 수요 증가에 따른 안정적인 성장 전망
- 핵심 장비인 웨이퍼 처리 및 패키징 장비 분야에서 선도적인 기술력 보유
- 국내외 주요 반도체 업체들과의 안정적 고객사 네트워크 및 거래 다변화
- 정부의 대규모 반도체 산업 육성 정책 수혜 기대
- 해외 시장 확대 및 현지 생산기지 구축으로 글로벌 경쟁력 향상

4. 리스크 요인
- 반도체 시장의 경기 변동에 따른 매출 및 이익 변동성
- 원자재 및 부품 가격 상승으로 인한 원가 부담 증가 가능성
- 경쟁 심화와 기술 혁신 속도의 지연 가능성
- 국제 무역 문제 및 규제 강화에 따른 해외 사업 리스크
- 환율 변동성으로 인한 이익 변동 위험
- 고객사 집중도 및 특정 거래처 의존도 문제
- 공급망 이슈 및 물류 지연으로 인한 생산 차질 가능성
- 환경 및 안전 관련 규제 강화로 인한 비용 상승

5. 재무 및 핵심 지표
최근 실적에서는 매출과 영업이익이 증가 추세를 보이며 수익성이 개선되고 있습니다. 부채비율과 유동비율 모두 안정적인 수준이며, ROE와 ROA 지표도 양호하여 재무 건전성이 확보되어 있습니다